# Step 4: Nearest-Neighbor Spatial Join for Unmatched Parcels

Russell Blessing

## Overview

This notebook performs a nearest-neighbor spatial join for parcels that did not receive a string match in Step 3. For each county, the unmatched OneMap parcel geometries are joined to the nearest CoreLogic point. Ties (a parcel equidistant from multiple CL points) are resolved by Levenshtein address similarity. The final output combines the string-matched and distance-matched records into a single parcel–CoreLogic table.

In [ ]:
library(sf)


Linking to GEOS 3.12.0, GDAL 3.11.0, PROJ 9.2.1; sf_use_s2() is TRUE


Attaching package: 'dplyr'

The following objects are masked from 'package:stats':

    filter, lag

The following objects are masked from 'package:base':

    intersect, setdiff, setequal, union

In [ ]:
out_dir <- "/proj/mhinolab/users/rbless/data/Obstacles_Output"

# CoreLogic (all character)
cl <- read_csv(
  file.path(out_dir, "cl.csv"),
  col_types = cols(.default = col_character())
)

# Previously string-matched parcel indices
stringmatched <- read_csv(
  file.path(out_dir, "parcel_cl_stringmatch.csv"),
  col_types = cols(.default = col_character())
) |>
  mutate(parcel_index = as.character(parcel_index))

# Read spatial parcels (keep geometry for the join)
nc1_attr <- st_read(
  file.path(out_dir, "parcels_study_area.gpkg"),
  query = "SELECT parcel_index, cntyfips FROM parcels_study_area",
  quiet = TRUE
) |>
  st_drop_geometry() |>
  mutate(
    parcel_index = as.character(parcel_index),
    cntyfips     = paste0("37", cntyfips)
  )

unmatched_ids <- nc1_attr |>
  filter(!parcel_index %in% stringmatched$parcel_index) |>
  pull(parcel_index)

# Read only unmatched rows with geometry using a SQL filter
unmatched_nc1 <- st_read(
  file.path(out_dir, "parcels_study_area.gpkg"),
  query = paste0(
    "SELECT * FROM parcels_study_area WHERE parcel_index IN (",
    paste(unmatched_ids, collapse = ","),
    ")"
  ),
  quiet = TRUE
) |>
  mutate(
    parcel_index = as.character(parcel_index),
    cntyfips     = paste0("37", cntyfips)
  )

rm(nc1_attr); gc()


            used   (Mb) gc trigger   (Mb)  max used   (Mb)
Ncells  47790859 2552.4   88811620 4743.1  62554291 3340.8
Vcells 570215346 4350.4  773537381 5901.7 640836481 4889.2

In [ ]:
# Reproject to NC State Plane (meters) for distance calculations
unmatched_nc1  <- st_transform(unmatched_nc1,  crs = 32119)
cl_unmatched   <- st_transform(cl_unmatched,   crs = st_crs(unmatched_nc1))

# Assign geometry IDs (factorize unique geometries)
unmatched_nc1 <- unmatched_nc1 |>
  mutate(geometry_id = as.integer(factor(as.character(st_geometry(unmatched_nc1)))))

cl_unmatched <- cl_unmatched |>
  mutate(geometry_id1 = as.integer(factor(as.character(st_geometry(cl_unmatched)))))


In [ ]:
# Save intermediate geopackages
st_write(unmatched_nc1, file.path(out_dir, "nc1geoms_distmatch.gpkg"),
         delete_dsn = TRUE, quiet = TRUE)

st_write(cl_unmatched, file.path(out_dir, "cl_wID.gpkg"),
         delete_dsn = TRUE, quiet = TRUE)


In [ ]:
# Drop to unique CL geometries before joining (avoids redundant distance calcs)
cl_unique <- cl_unmatched |>
  distinct(geometry_id1, .keep_all = TRUE)

# County-by-county nearest-neighbor join
nn_results <- map(county_list, function(fips) {
  cl_fips      <- cl_unique      |> filter(`FIPS CODE` == fips)
  parcels_fips <- unmatched_nc1  |> filter(cntyfips    == fips)

  if (nrow(cl_fips) == 0 || nrow(parcels_fips) == 0) return(NULL)

  # Nearest neighbour join (st_join with st_nearest_feature gives distances
  # via st_distance; use nngeo for the distance column directly)
  nn <- st_join(parcels_fips, cl_fips, join = st_nearest_feature)

  # Compute actual distance to the matched point
  nn_geom <- st_geometry(nn)
  nn <- nn |>
    mutate(distance = as.numeric(
      st_distance(
        nn_geom,
        cl_fips$geometry[match(geometry_id1, cl_fips$geometry_id1)],
        by_element = TRUE
      )
    ))

  # For each parcel, keep only the minimum-distance match(es)
  nn |>
    st_drop_geometry() |>
    group_by(parcel_index) |>
    filter(distance == min(distance)) |>
    ungroup()
}, .progress = TRUE) |>
  list_rbind()


 ■■■                                8% |  ETA: 42s

 ■■■■■                             12% |  ETA: 49s

 ■■■■■                             13% |  ETA:  1m

 ■■■■■■                            17% |  ETA:  3m

 ■■■■■■■                           18% |  ETA:  2m

...

 ■■■■■■■■■■■■■■■■■■■■■■■■■■■       88% |  ETA:  1m

 ■■■■■■■■■■■■■■■■■■■■■■■■■■■■      89% |  ETA:  1m

 ■■■■■■■■■■■■■■■■■■■■■■■■■■■■■     92% |  ETA: 42s

 ■■■■■■■■■■■■■■■■■■■■■■■■■■■■■     95% |  ETA: 28s

 ■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■    97% |  ETA: 14s

In [ ]:
# Parcels that matched multiple CL geometries at equal minimum distance
tie_counts      <- count(nn_results, parcel_index)
tie_parcelids   <- tie_counts$parcel_index[tie_counts$n > 1]

duplicated_rows <- nn_results |> filter(parcel_index %in%  tie_parcelids)
cl_dist_nosort  <- nn_results |> filter(!parcel_index %in% tie_parcelids)

write_csv(duplicated_rows, file.path(out_dir, "parcel_joins_to_sort.csv"))
write_csv(cl_dist_nosort,  file.path(out_dir, "parcel_cl_dist_joins_nosort.csv"))


In [ ]:
# CL attribute table (no geometry) for address lookup
cl_attr <- cl_unmatched |> st_drop_geometry()

# One row per tied parcel (the nc1 side)
nc1_pcls <- duplicated_rows |> distinct(parcel_index, .keep_all = TRUE)

set.seed(987)

find_best_cl_match <- function(pc_row) {
  pc       <- nc1_pcls[pc_row, ]
  cl_geoms <- duplicated_rows |>
    filter(parcel_index == pc$parcel_index) |>
    pull(geometry_id1)

  cl_pcls <- cl_attr |> filter(geometry_id1 %in% cl_geoms)

  # Choose address field: situs preferred, mailing as fallback
  if (!is.na(pc$siteadd) && pc$siteadd != "") {
    candidate_addresses <- paste(cl_pcls$`SITUS HOUSE NUMBER`,
                                 cl_pcls$`SITUS STREET NAME`)
    query_address       <- pc$siteadd
  } else {
    candidate_addresses <- paste(cl_pcls$`MAILING HOUSE NUMBER`,
                                 cl_pcls$`MAILING STREET NAME`)
    query_address       <- pc$mailadd
  }

  scores    <- stringdist(as.character(query_address), candidate_addresses,
                          method = "lv")
  best_idx  <- which(scores == min(scores))

  # Break remaining ties randomly
  if (length(best_idx) > 1) best_idx <- sample(best_idx, 1)

  cl_pcls[best_idx, ]
}

matched_cls  <- map(seq_len(nrow(nc1_pcls)), find_best_cl_match)
cl_dist_sorted <- bind_cols(
  nc1_pcls |> select(-any_of(names(cl_attr))),   # nc1 columns only
  list_rbind(matched_cls)                          # winning CL row
)

write_csv(cl_dist_sorted, file.path(out_dir, "cl_dist_sorted.csv"))


In [ ]:
cl_string <- read_csv(
  file.path(out_dir, "parcel_cl_stringmatch.csv"),
  col_types = cols(.default = col_character())
)

# Stack string-matched, distance-matched (no sort needed), distance-matched (sorted)
pcl_comb <- bind_rows(
    cl_string      |> mutate(across(everything(), as.character)),
    cl_dist_nosort |> mutate(across(everything(), as.character)),
    cl_dist_sorted |> mutate(across(everything(), as.character))
  ) |>
  mutate(geometry_id = ifelse(is.na(geometry_id), "", as.character(geometry_id))) |>
  arrange(parcel_index, geometry_id) |>                # blank geometry_id sorts first
  distinct(parcel_index, .keep_all = TRUE)             # prefer string match

write_csv(pcl_comb, file.path(out_dir, "nc1_cl_merge.csv"))


## Summary

In [ ]:
n_string   <- nrow(cl_string)
n_nosort   <- nrow(cl_dist_nosort)
n_sorted   <- nrow(cl_dist_sorted)
n_total    <- nrow(pcl_comb)
n_unmatched_input <- nrow(unmatched_nc1)


In [ ]:
tibble::tibble(
  Source  = c("String-matched (Step 3)", "Distance-matched (no ties)",
              "Distance-matched (ties resolved)", "Total unique parcels in merge"),
  Records = c(n_string, n_nosort, n_sorted, n_total)
) |>
  gt() |>
  tab_header(title = "Parcel–CoreLogic Merge Summary") |>
  fmt_integer(columns = Records) |>
  cols_label(Source = "", Records = "N")


-   **534,946** parcels entered the spatial join (no string match found in Step 3).
-   **534,946** were unambiguously distance-matched.
-   **0** had tied distances and were resolved by Levenshtein address similarity.
-   The combined file `nc1_cl_merge.csv` contains **4,273,305** unique parcel records.